In [ ]:
%cd ..

In [2]:
from dotenv import load_dotenv

load_dotenv()


True

In [ ]:
import sys
import os

sys.path.insert(0, r"C:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions")

In [3]:
import os
import sys
import json
import time
import requests
import pandas as pd
import pyarrow as pa
from dateutil import parser
import pyarrow.parquet as pq
from datetime import datetime


In [4]:

if not os.path.exists("./tmp/data"):
    os.makedirs("./tmp/data")


In [ ]:
from common.config import *
from common.http_util import *
from common.crawler_util import *
from common.ambari_util import *


def fetch_resource_name_freshwork(resource_name, records, **kwargs):
    print("Start crawl : ", resource_name)
    start_time = time.time()

    HDFS_BASE = "s3a://vcs-raw/hr-raw"
    # STATE_PATH = rc["state_path"]
    # BASE_URL = None
    # RESOURCE_URL = None
    # API_KEY_PATH = rc["api_key_path"]
    # API_COOKIE_PATH = rc["api_cookie_path"]
    # QUERY_PARAMS = None
    ENABLE_STATE =False
    HIVE_DB = "hr_raw"
    # crawl_mode = "modified_and_new"
    crawl_mode = kwargs.get("crawl_mode", "static")
    schema_local_path = None
    # result_json_key = "deleted_deals"

    print(resource_name)
    start_time = time.time()
    # =========================
    # MAIN
    # =========================
    if ENABLE_STATE:
        last_state = read_last_state(resource_name)
        print("Last state =", last_state)
    else:
        last_state = None

    if not records:
        print("No new data")
        out_of_data = True
        return True

    # =========================
    # Pandas → Parquet
    # =========================

    now = datetime.now()
    partition_path = "{}/{}".format(HDFS_BASE, resource_name)

    filename = "data_{}_{}{:02d}{:02d}_{}{:02d}{:02d}.parquet".format(
        resource_name, now.year, now.month, now.day, now.hour, now.minute, now.second
    )
    local_parquet = "./tmp/data/hr_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_parquet), exist_ok=True)

    # df.to_parquet(local_parquet,engine="pyarrow", compression="snappy", index=False)
    records = convert_json_add_ts_columns(records)

    schema_tm_path = "./resources/parquet_schema/hr_raw/{}.json".format(resource_name)
    schema = None
    if schema_local_path:
        schema = load_pyarrow_schema_from_json(schema_local_path)

    if os.path.exists(schema_tm_path):
        schema = load_pyarrow_schema_from_json(schema_tm_path)

    if not schema:
        schema = infer_schema_from_json(records)
        schema_json = save_pyarrow_type_to_json(schema)
        write_file_json(schema_tm_path, schema_json)

    records = convert_json_list_by_arrow_schema(records, schema)

    df = pd.DataFrame(records)
    data_table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)

    pq.write_table(data_table, local_parquet, compression="snappy")

    if crawl_mode == "static":
        replace_hdfs_https("{}".format(partition_path), local_parquet)
    else:
        upload_hdfs_https("{}".format(partition_path), local_parquet)

    print("Uploaded parquet to", partition_path)

    # =========================
    # Generate SQL (TEXT ONLY)
    # =========================
    sql = gen_spark_create_table(
        schema=schema,
        db=HIVE_DB,
        table=resource_name,
        location="{}/{}".format(HDFS_BASE, resource_name),
    )

    # filename = "create_table_{}_{}{:02d}{:02d}.sql".format(
    #     resource_name,
    #     now.hour,
    #     now.minute,
    #     now.second
    # )

    filename = "create_table_{}.sql".format(resource_name)

    local_sql = "./tmp/data/hr_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_sql), exist_ok=True)

    with open(local_sql, "w") as f:
        f.write(sql)

    # upload_hdfs_https(
    #     "{}/{}".format(HDFS_BASE, resource_name),
    #     local_sql
    # )

    print("Uploaded SQL definition")

    # =========================
    # Save new state
    # =========================
    if ENABLE_STATE and "updated_at" in df.columns:
        max_ts = get_max_updated_at_str(df)
        print("last state ", resource_name, "max_ts=", max_ts)
        max_ts = subtract_minutes(max_ts, 30)
        write_last_state(max_ts, resource_name)
        print("last state ", resource_name, "max_ts=", max_ts)

    elapsed = time.time() - start_time
    print("Loop {} took {:.3f}s".format(resource_name, elapsed))
    if len(records) < 100:
        print("No new data")
        out_of_data = True
        return True
    return False


In [6]:
import pandas as pd
import re
from collections import defaultdict
def excel_col_name(idx: int) -> str:
    """Zero-based index → Excel column (A, B, ..., AA)"""
    name = ""
    while idx >= 0:
        idx, rem = divmod(idx, 26)
        name = chr(rem + ord("A")) + name
        idx -= 1
    return name


def snake_case(text: str) -> str:
    text = (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

def read_excel_and_normalize_columns(
    file_path: str,
    mapping: dict,
    sheet_name=0,
    header_row=0,
    drop_rows=2
) -> pd.DataFrame:
    # đọc raw, chưa set header
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=None, dtype=str)

    raw_headers = df.iloc[header_row]
    seen = defaultdict(int)
    new_columns = []

    for idx, col in enumerate(raw_headers):
        if pd.isna(col) or col == "":
            base = "nan"
        else:
            # 1️⃣ ưu tiên dictionary
            base = mapping.get(col)

            # 2️⃣ fallback snake_case
            if base is None:
                print(f"Not found for col = '{col}'")
                base = snake_case(col)

        seen[base] += 1

        if seen[base] > 1:
            excel_col = excel_col_name(idx).lower()
            base = f"{base}__{excel_col}"

        new_columns.append(base)

    # gán columns sạch
    df.columns = new_columns

    # drop header rows
    df = df.iloc[drop_rows:].reset_index(drop=True)

    return df



In [7]:

COLUMN_DICT  = {
  "Mã nhân viên": "employee_id",
  "Họ tên": "employee_name",
  "Trạng thái hiện tại": "status",
  "Trạng thái tuyển mới": "hire_status",
  "Đối tượng": "employee_group",

  "Diện hợp đồng\n(công ty độc lập)": "contract_company_type",
  "Diện cũ\n(trước công ty độc lập)": "previous_independent_company_type",

  "Chi nhánh": "branch_name",
  "Khối (N)": "division_name",
  "Đơn vị (N-1)": "n1_group",
  "Bộ phận (N-2)": "n2_group",
  "Bộ phận (N-3)": "n3_group",

  "Role Base": "job_title",
  "Level": "employee_level",
  "Vùng": "regional_staff",
  "Phân cấp quản lý": "management_level",

  "Hire Date": "hire_date",
  "Hire Date VCS": "hire_date_vcs",
  "Ngày nghỉ việc": "termination_date",

  "Business Email": "business_email",
  "DOB": "dob",
  "Giới tính": "gender",
  "Nhóm thâm niên": "tenure_group",

  "PR Q1/2024": "pr_q1_2024",
  "PR Q2/2024": "pr_q2_2024",
  "PR Q3/2024": "pr_q3_2024",
  "PR Q4/2024": "pr_q4_2024",
  "TB 2024": "pr_avg_2024",

  "PR Q1/2025": "pr_q1_2025",
  "PR Q2/2025": "pr_q2_2025",
  "PR Q3/2025": "pr_q3_2025",

  "Danh hiệu 2024": "achievement_title_2024",
  "9box - Nhóm Talent": "talent_group",
  "Nhận xét của CBQL": "manager_comment",

  "Trạng thái nghỉ": "termination_status",
  "Phân loại lý do nghỉ 1": "termination_reason_1",
  "Phân loại lý do nghỉ 2": "termination_reason_2",
  "Chi tiết lý do nghỉ việc": "termination_detail_reason",
  "Thông tin nghỉ việc bổ sung từ HR/CBQL": "termination_additional_info",

  "Đề xuất": "recommendation",
  "Khía cạnh nhân sự đánh giá cao ở VCS": "high_potential_factor",
  "Nơi làm việc tiếp theo": "next_workplace",
  "Người exit interview": "exit_interviewer",

  "Tháng bắt đầu": "start_month",
  "Tháng kết thúc": "end_month",

  "T1": "t1",
  "T2": "t2",
  "T3": "t3",
  "T4": "t4",
  "T5": "t5",
  "T6": "t6",
  "T7": "t7",
  "T8": "t8",
  "T9": "t9",
  "T10": "t10",
  "T11": "t11",
  "T12": "t12",

  "Tuyển mới": "is_new_hire",
  "Out chủ động": "voluntary_exit",
  "VCS cho nghỉ": "company_termination",
  "Tăng mới\n+ Out chủ động": "new_hire_and_voluntary_exit",
  "Tăng mới + VCS cho nghỉ": "new_hire_and_company_termination",

  "Q1": "q1",
  "Q2": "q2",
  "Q3": "q3",
  "Q4": "q4",

  "Độ tuổi": "hire_age",
  "Portfolio SPDV": "portfolio",
  "Job Family": "job_family",

  "Vị trí khung": "core_position",

  "TB PR Quý 1,2/2025": "pr_avg_q_1_2_2025",
  "Số kỳ PR Gần đạt/Không đạt (Từ Quý III/2024 - Quý II/2025)": "pr_near_fail_count_q3_2024_q2_2025",
  "Số kỳ PR Gần đạt/Không đạt (Từ Quý I, II/2025)": "pr_near_fail_count_q1_q2_2025",
  "Số kỳ PR vượt yêu cầu (Từ Quý III/2024 - Quý II/2025)": "pr_exceed_count_q3_2024_q2_2025",
  "Số kỳ PR vượt yêu cầu (Quý I, II/2025)": "pr_exceed_count_q1_q2_2025",

  "Kết quả CP đợt 1 2025": "cp_result_round_1_2025",
  "Đề xuất CP đợt 2 2025": "cp_recommendation_round_2_2025",

  "Số lần được khen thưởng CNXS": "recognition_count",
  "Khen thưởng CNXS 6 tháng": "recognition_6_months",

  "Nhân sự được chú trọng phát triển": "is_development_focus_employee",
  "Nhân sự được ghi nhận đúng": "is_properly_recognized_employee",
  "Nhân sự PR tốt không có ghi nhận": "high_pr_without_recognition_flag",
  "PR không tốt, được ghi nhận": "low_pr_but_recognized_flag",

  "Khung/Lõi": "is_core_employee",
  "Tuổi": "age",
  "Key": "key",
  "HP": "hp",
  "Key/HP": "key_hp",

  None: "unknown01"
}

NEW_COLUMN_MAPPING = {
    # ===== Core info =====
    "Ma_nhan_vien": "employee_id",
    "Ho_ten": "employee_name",
    "Trang_thai_lam_viec": "status",
    "Trang_thai_tuyen_moi": "hire_status",
    "Dien_doi_tuong": "employee_group",
    "Dien_hop_dong": "contract_company_type",

    "Chi_nhanh": "branch_name",
    "Khoi_N": "division_name",
    "Don_vi_N-1": "n1_group",
    "Bo_phan_N-2": "n2_group",
    "Bo_phan_N-3": "n3_group",

    "Vi_tri_cong_viec": "job_title",
    "Level": "employee_level",
    "Vung": "regional_staff",
    "Phan_cap": "management_level",

    # ===== Dates =====
    "Ngay_hieu_luc_level": "level_effective_date",  # NEW
    "Ngay_gia_nhap_Viettel": "hire_date_viettel",   # NEW
    "Ngay_gia_nhap_VCS": "hire_date_vcs",
    "Ngay_nghi_viec": "termination_date",

    # ===== Personal =====
    "Email": "business_email",
    "Ngay_sinh": "dob",
    "Gioi_tinh": "gender",
    "Mobile": "mobile_phone",  # NEW

    # ===== Talent / classification =====
    "Nhom_Loi_Khung": "is_core_employee",
    "Nhom_Key": "key",
    "Nhom_HP": "hp",

    # ===== Performance (MSN ~ PR) =====
    "Diem_MSN_Q1.2024": "pr_q1_2024",
    "Diem_MSN_Q2.2024": "pr_q2_2024",
    "Diem_MSN_Q3.2024": "pr_q3_2024",
    "Diem_MSN_Q4.2024": "pr_q4_2024",

    "Diem_MSN_Q1.2025": "pr_q1_2025",
    "Diem_MSN_Q2.2025": "pr_q2_2025",
    "Diem_MSN_Q3.2025": "pr_q3_2025",
    "Diem_MSN_Q4.2025": "pr_q4_2025",  # NEW

    # ===== 2026 (NEW hoàn toàn) =====
    "Diem_MSN_Q1.2026": "pr_q1_2026",
    "Diem_MSN_Q2.2026": "pr_q2_2026",
    "Diem_MSN_Q3.2026": "pr_q3_2026",
    "Diem_MSN_Q4.2026": "pr_q4_2026",

    # ===== KI (NEW metric) =====
    "KI_Q1.2025": "ki_q1_2025",
    "KI_Q2.2025": "ki_q2_2025",
    "KI_Q3.2025": "ki_q3_2025",
    "KI_Q4.2025": "ki_q4_2025",
    "KI_N2025": "ki_2025",

    "KI_Q1.2026": "ki_q1_2026",
    "KI_Q2.2026": "ki_q2_2026",
    "KI_Q3.2026": "ki_q3_2026",
    "KI_Q4.2026": "ki_q4_2026",
    "KI_N2026": "ki_2026",

    # ===== Ranking =====
    "Performance_Ranking": "performance_ranking",  # NEW
    "Potential_Ranking": "potential_ranking",      # NEW
    "9box_Ranking": "talent_group",

    # ===== Reward =====
    "Khen_thuong_2024": "achievement_title_2024",
    "Khen_thuong_2025": "achievement_title_2025",  # NEW
    "Khen_thuong_2026": "achievement_title_2026",  # NEW

    # ===== Exit =====
    "Trang_thai_nghi_viec": "termination_status",
    "Phan_loai_ly_do_nghi_viec_1": "termination_reason_1",
    "Phan_loai_ly_do_nghi_viec_2": "termination_reason_2",
    "Ly_do_nghi_viec": "termination_detail_reason",
    "Noi_lam_viec_tiep_theo": "next_workplace",

    # ===== Demographic =====
    "Tham_nien": "tenure",       # NEW
    "So_tuoi": "age",
    "Nhom_tuoi": "age_group",   # NEW

    # ===== Time =====
    "Thang_bat_dau": "start_month",
    "Thang_ket_thuc": "end_month",

    # ===== Month flags =====
    "T1": "t1",
    "T2": "t2",
    "T3": "t3",
    "T4": "t4",
    "T5": "t5",
    "T6": "t6",
    "T7": "t7",
    "T8": "t8",
    "T9": "t9",
    "T10": "t10",
    "T11": "t11",
    "T12": "t12",

    # ===== Movement =====
    "Tang_moi": "is_new_hire",
    "Nghi_chu_dong": "voluntary_exit",
    "Thai_loai": "company_termination",
    "Tang_moi_nghi_chu_dong": "new_hire_and_voluntary_exit",
    "Tang_moi_thai_loai": "new_hire_and_company_termination",
}



In [69]:
COLUMN_DICT = COLUMN_DICT | NEW_COLUMN_MAPPING

In [67]:
# COLUMN_DICT

In [ ]:
# filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\hr-data\xlsx\VCS_Workforce Report 2025_31122025.xlsx"
# df = read_excel_and_normalize_columns(
#       filename,
#     sheet_name="Data_MHTC Mới",
#     mapping=COLUMN_DICT,
#     header_row=1,
# )

# filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\hr-data\xlsx\VCS_Workforce Report 2026_09032026.xlsx"
# df = read_excel_and_normalize_columns(
#       filename,
#     sheet_name="Active_2026",
#     mapping=COLUMN_DICT,
#     header_row=1,
# )



In [79]:
df.head(1)

,employee_id,employee_name,status,hire_status,employee_group,contract_company_type,branch_name,division_name,n1_group,n2_group,...,is_development_focus_employee,is_properly_recognized_employee,high_pr_without_recognition_flag,low_pr_but_recognized_flag,is_core_employee,age,key,hp,key_hp,is_core_employee__ea
0,085840,Nguyễn Sơn Hải,Active,NaN,TDS,TDS,VCS - HN,Khối Lãnh đạo,Ban Giám đốc,Thủ trưởng đơn vị,...,NaN,NaN,NaN,NaN,x,43.48888888888889,x,NaN,x,Mức 1_ Nhóm Lõi


In [73]:
# for c in df.columns:
#     print(c)

In [80]:
agg_dict = {col: "first" for col in df.columns}
agg_dict["status"] = "last"

df_result = (
    df
    .groupby("employee_id", as_index=False)
    .agg(agg_dict)
)

In [81]:
import json
df_result = df_result.fillna("")
df_result["filename"]= filename.split("\\")[-1]

# 6. Chuyển sang JSON
records = df_result.to_dict(orient="records")

with open("hr_employee_onboard.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print("✅ Done: hr_employee_onboard.json created")


✅ Done: hr_employee_onboard.json created


In [76]:
fetch_resource_name_freshwork("hr_employee_onboard", records, crawl_mode="static")

Start crawl :  hr_employee_onboard
hr_employee_onboard
Replace Upload  /opt/datasets/crawlers/vcs/human-resource/data/hr_employee_onboard ./tmp/data/hr_employee_onboard/data_hr_employee_onboard_20260318_200720.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to /opt/datasets/crawlers/vcs/human-resource/data/hr_employee_onboard
Uploaded SQL definition
Loop hr_employee_onboard took 8.451s


False

In [82]:
fetch_resource_name_freshwork("hr_employee_onboard_logs", records, crawl_mode="modified_and_new")

Start crawl :  hr_employee_onboard_logs
hr_employee_onboard_logs
Add Upload  /opt/datasets/crawlers/vcs/human-resource/data/hr_employee_onboard_logs ./tmp/data/hr_employee_onboard_logs/data_hr_employee_onboard_logs_20260318_200847.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to /opt/datasets/crawlers/vcs/human-resource/data/hr_employee_onboard_logs
Uploaded SQL definition
Loop hr_employee_onboard_logs took 10.187s


False

In [83]:
COLUMN_DICT_OUT= {
  "Mã nhân viên": "employee_code",
  "Họ tên": "full_name",
  "Trạng thái hiện tại": "current_status",
  "Trạng thái tuyển mới": "new_hire_status",
  "Đối tượng": "employee_object",
  "Diện hợp đồng": "contract_type",
  "(công ty độc lập)": "independent_company_flag",

  "Diện hợp đồng\n(công ty độc lập)": "contract_type_independent_company",

  "Chi nhánh": "branch",
  "Khối (N)": "division_n",
  "Đơn vị (N-1)": "unit_n_1",
  "Bộ phận (N-2)": "department_n_2",

  "Role Base": "role_base",
  "Level": "level",
  "Vùng": "region",
  "Phân cấp quản lý": "management_level",

  "Diện cũ\n(trước công ty độc lập)": "previous_contract_type_before_independent",

  "Hire Date": "hire_date",
  "Hire Date VCS": "hire_date_vcs",
  "Ngày nghỉ việc": "resigned_date",

  "Business Email": "business_email",
  "DOB": "date_of_birth",
  "Giới tính": "gender",
  "Nhóm thâm niên": "seniority_group",

  "Trạng thái nghỉ": "resignation_status",
  "Phân loại lý do nghỉ 1": "resignation_reason_level_1",
  "Phân loại lý do nghỉ 2": "resignation_reason_level_2",
  "Chi tiết lý do nghỉ việc": "resignation_reason_detail",
  "Thông tin nghỉ việc bổ sung từ HR/CBQL": "resignation_additional_info",

  "Đề xuất": "recommendation",
  "Khía cạnh nhân sự đánh giá cao ở VCS": "hr_strengths_at_vcs",
  "Nơi làm việc tiếp theo": "next_workplace",
  "Người exit interview": "exit_interviewer",

  "Vị trí khung": "job_framework_position",
  "Key/HP": "key_hp_flag",
  "Note": "note",
  "PR Q1/2024": "pr_q1_2024",
  "PR Q2/2024": "pr_q2_2024",
  "PR Q3/2024": "pr_q3_2024",
  "PR Q4/2024": "pr_q4_2024",

  "TB 2024": "pr_avg_2024",

  "PR Q1/2025": "pr_q1_2025",
  "PR Q2/2025": "pr_q2_2025",

  "TB PR Quý 1,2/2025": "pr_avg_q1_q2_2025",

  "Danh hiệu 2024": "award_2024",

  "9box - Nhóm Talent": "talent_9box_group",

  "Nhận xét của CBQL": "manager_comment",

  "Số kỳ PR Gần đạt/Không đạt (Từ Quý III/2024 - Quý II/2025)": "pr_near_or_not_met_count_q3_2024_to_q2_2025",

  "Số kỳ PR Gần đạt/Không đạt (Từ Quý I, II/2025)": "pr_near_or_not_met_count_q1_q2_2025",

  "Mã nhân viên ảo": "virtual_employee_code",
}


In [85]:
NEW_COLUMN_MAPPING_OUT = {
    # ===== Core =====
    "Ma_nhan_vien": "employee_code",
    "Ho_ten": "full_name",
    "Trang_thai_lam_viec": "current_status",
    "Trang_thai_tuyen_moi": "new_hire_status",
    "Dien_doi_tuong": "employee_object",
    "Dien_hop_dong": "contract_type",

    "Chi_nhanh": "branch",
    "Khoi_N": "division_n",
    "Don_vi_N-1": "unit_n_1",
    "Bo_phan_N-2": "department_n_2",
    "Bo_phan_N-3": "department_n_3",  # NEW

    "Vi_tri_cong_viec": "role_base",
    "Level": "level",
    "Vung": "region",
    "Phan_cap": "management_level",

    # ===== Dates =====
    "Ngay_hieu_luc_level": "level_effective_date",   # NEW
    "Ngay_gia_nhap_Viettel": "hire_date_viettel",    # NEW
    "Ngay_gia_nhap_VCS": "hire_date_vcs",
    "Ngay_nghi_viec": "resigned_date",

    # ===== Personal =====
    "Email": "business_email",
    "Ngay_sinh": "date_of_birth",
    "Gioi_tinh": "gender",
    "Mobile": "mobile_phone",  # NEW

    # ===== Talent =====
    "Nhom_Loi_Khung": "job_framework_position",
    "Nhom_Key": "key_flag",     # NEW (tách từ key_hp_flag)
    "Nhom_HP": "hp_flag",       # NEW

    # ===== Performance (MSN → PR) =====
    "Diem_MSN_Q1.2024": "pr_q1_2024",
    "Diem_MSN_Q2.2024": "pr_q2_2024",
    "Diem_MSN_Q3.2024": "pr_q3_2024",
    "Diem_MSN_Q4.2024": "pr_q4_2024",

    "Diem_MSN_Q1.2025": "pr_q1_2025",
    "Diem_MSN_Q2.2025": "pr_q2_2025",
    "Diem_MSN_Q3.2025": "pr_q3_2025",  # NEW
    "Diem_MSN_Q4.2025": "pr_q4_2025",  # NEW

    # ===== 2026 =====
    "Diem_MSN_Q1.2026": "pr_q1_2026",
    "Diem_MSN_Q2.2026": "pr_q2_2026",
    "Diem_MSN_Q3.2026": "pr_q3_2026",
    "Diem_MSN_Q4.2026": "pr_q4_2026",

    # ===== KI (NEW metric) =====
    "KI_Q1.2025": "ki_q1_2025",
    "KI_Q2.2025": "ki_q2_2025",
    "KI_Q3.2025": "ki_q3_2025",
    "KI_Q4.2025": "ki_q4_2025",
    "KI_N2025": "ki_2025",

    "KI_Q1.2026": "ki_q1_2026",
    "KI_Q2.2026": "ki_q2_2026",
    "KI_Q3.2026": "ki_q3_2026",
    "KI_Q4.2026": "ki_q4_2026",
    "KI_N2026": "ki_2026",

    # ===== Ranking =====
    "Performance_Ranking": "performance_ranking",  # NEW
    "Potential_Ranking": "potential_ranking",      # NEW
    "9box_Ranking": "talent_9box_group",

    # ===== Reward =====
    "Khen_thuong_2024": "award_2024",
    "Khen_thuong_2025": "award_2025",  # NEW
    "Khen_thuong_2026": "award_2026",  # NEW

    # ===== Exit =====
    "Trang_thai_nghi_viec": "resignation_status",
    "Phan_loai_ly_do_nghi_viec_1": "resignation_reason_level_1",
    "Phan_loai_ly_do_nghi_viec_2": "resignation_reason_level_2",
    "Ly_do_nghi_viec": "resignation_reason_detail",
    "Noi_lam_viec_tiep_theo": "next_workplace",

    # ===== Demographic =====
    "Tham_nien": "seniority",       # NEW (khác seniority_group)
    "So_tuoi": "age",               # NEW
    "Nhom_tuoi": "age_group",       # NEW
}

In [86]:
COLUMN_DICT_OUT = COLUMN_DICT_OUT | NEW_COLUMN_MAPPING_OUT

In [ ]:


# filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\hr-data\xlsx\VCS_Workforce Report 2025_31122025.xlsx"
# df = read_excel_and_normalize_columns(
#       filename,
#     sheet_name="Out_2025",
#     mapping=COLUMN_DICT_OUT,
#     header_row=1,
#     drop_rows=2,
# )


# filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\hr-data\xlsx\VCS_Workforce Report 2026_09032026.xlsx"
# df = read_excel_and_normalize_columns(
#       filename,
#     sheet_name="Out_2026",
#     mapping=COLUMN_DICT_OUT,
#     header_row=1,
# )


In [95]:
df.head(1)

,employee_code,full_name,current_status,new_hire_status,employee_object,contract_type_independent_company,branch,division_n,unit_n_1,department_n_2,...,nan__dj,nan__dk,nan__dl,nan__dm,nan__dn,nan__do,nan__dp,nan__dq,nan__dr,nan__ds
0,807594,Phạm Khánh Ly,Nghỉ việc (out chủ động),NaN,NDS,Vilado,VCS - HN,Khối Cơ quan,Phòng Tổ chức Hành chính,BP Nhân sự,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [96]:
agg_dict = {col: "first" for col in df.columns}
agg_dict["current_status"] = "last"

df_result = (
    df
    .groupby("employee_code", as_index=False)
    .agg(agg_dict)
)

In [97]:
import json
df_result = df_result.fillna("")

# 6. Chuyển sang JSON
records = df_result.to_dict(orient="records")

with open("hr_employee_resigned.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print("✅ Done: hr_employee_resigned.json created")


✅ Done: hr_employee_resigned.json created


In [91]:
fetch_resource_name_freshwork("hr_employee_resigned", records, crawl_mode="static")

Start crawl :  hr_employee_resigned
hr_employee_resigned
Replace Upload  /opt/datasets/crawlers/vcs/human-resource/data/hr_employee_resigned ./tmp/data/hr_employee_resigned/data_hr_employee_resigned_20260318_200935.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to /opt/datasets/crawlers/vcs/human-resource/data/hr_employee_resigned
Uploaded SQL definition
Loop hr_employee_resigned took 1.112s
No new data


True

In [98]:
fetch_resource_name_freshwork("hr_employee_resigned_logs", records, crawl_mode="modified_and_new")

Start crawl :  hr_employee_resigned_logs
hr_employee_resigned_logs
Add Upload  /opt/datasets/crawlers/vcs/human-resource/data/hr_employee_resigned_logs ./tmp/data/hr_employee_resigned_logs/data_hr_employee_resigned_logs_20260318_201001.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to /opt/datasets/crawlers/vcs/human-resource/data/hr_employee_resigned_logs
Uploaded SQL definition
Loop hr_employee_resigned_logs took 2.322s


False

In [8]:
HEADCOUNT_MAPPING = {
    # ===== ID / Meta =====
    "idx": "row_index",                  # NEW
    "Dinh_bien": "headcount_plan",       # NEW
    "Hien_trang_dinh_bien": "headcount_status",  # NEW

    # ===== Org structure =====
    "Khoi_N": "division_n",
    "Don_vi_N-1": "unit_n_1",
    "Bo_phan_N-2": "department_n_2",
    "Bo_phan_N-3": "department_n_3",     # NEW

    "Chi_nhanh": "branch",

    # ===== Job =====
    "Vi_tri_cong_viec": "role_base",
    "Mang_chuyen_huong": "specialization",   # NEW
    "Level_yeu_cau": "required_level",       # NEW
    "Trang_thai_vi_tri": "position_status",  # NEW

    # ===== Employee info =====
    "Ho_ten": "full_name",
    "Ma_nhan_vien": "employee_code",

    "Dien_doi_tuong": "employee_object",
    "Dien_hop_dong": "contract_type",

    "Ngay_gia_nhap_VCS": "hire_date_vcs",
    "Level": "level",
    "Vung": "region",
    "Phan_cap": "management_level",

    "Email": "business_email",

    # ===== Talent =====
    "Nhom_Loi_Khung": "job_framework_position",
    "Nhom_Loi_Khung_2": "job_framework_position_2",  # NEW

    # ===== Product / domain grouping =====
    "Nhom_SPDV": "service_group",          # NEW
    "Nhom_SPDV_moi": "service_group_new",  # NEW

    # ===== Strategic flags (boolean-like) =====
    "Nhom_AI": "is_ai_group",
    "Bo_may_AI": "is_ai_org",

    "Thi_truong_Phil_Nhat": "is_philippines_japan_market",
    "Bo_may_kinh_doanh_trong_nuoc": "is_domestic_business",
    "Bo_may_kinh_doanh_quoc_te": "is_international_business",
    "Bo_may_R&D": "is_rnd",

    "Nhom_gian_tiep": "is_indirect_group",
    "Nhom_ho_tro_kinh_doanh": "is_business_support",
    ' ':'n_a'
}

In [9]:
filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\hr-data\xlsx\VCS_Workforce Report 2026_09032026.xlsx"
df = read_excel_and_normalize_columns(
      filename,
    sheet_name="NhuCau_2026",
    mapping=HEADCOUNT_MAPPING,
    header_row=1,
)

Not found for col = ' '


In [108]:
# agg_dict = {col: "first" for col in df.columns}
# agg_dict["current_status"] = "last"

# df_result = (
#     df
#     .groupby("employee_code", as_index=False)
#     .agg(agg_dict)
# )

In [11]:
import json
df = df.fillna("")

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")

with open("hr_employee_headcount.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print("✅ Done: hr_employee_headcount.json created")


✅ Done: hr_employee_headcount.json created


In [13]:
fetch_resource_name_freshwork("hr_employee_headcount", records, crawl_mode="static")

Start crawl :  hr_employee_headcount
hr_employee_headcount
Replace Upload  /opt/datasets/crawlers/vcs/human-resource/data/hr_employee_headcount ./tmp/data/hr_employee_headcount/data_hr_employee_headcount_20260320_214055.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to /opt/datasets/crawlers/vcs/human-resource/data/hr_employee_headcount
Uploaded SQL definition
Loop hr_employee_headcount took 4.416s


False

In [14]:
fetch_resource_name_freshwork("hr_employee_headcount_logs", records, crawl_mode="static")

Start crawl :  hr_employee_headcount_logs
hr_employee_headcount_logs
Replace Upload  /opt/datasets/crawlers/vcs/human-resource/data/hr_employee_headcount_logs ./tmp/data/hr_employee_headcount_logs/data_hr_employee_headcount_logs_20260320_214107.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to /opt/datasets/crawlers/vcs/human-resource/data/hr_employee_headcount_logs
Uploaded SQL definition
Loop hr_employee_headcount_logs took 4.564s


False